# Bitcoin Price Forecasting — Data Science Course Project

## Educational pilot study created in 2024

This notebook is an English portfolio edition of the original course work. It explores a historical five-day-ahead Bitcoin price forecasting task using time-series and machine-learning approaches.

**Important:** This notebook is retained as a transparent learning record. Its original data files are not bundled, and its historical outputs have been removed rather than presented as current results. It is not a trading system, financial product, or source of financial advice.

For the evidence boundaries, methodology limitations, and a responsible refresh path, see the repository README and `LIMITATIONS.md`.


# TABLE OF CONTENTS

[1. IMPORTING LIBRARIES](#sec-1)

[2. DATA LOADING](#sec-2)

[3. DATA CLEANING](#sec-3)

* [3.1. Undestend Data & Merge DataSet](#sub-3-1)

* [3.2. Check and Input Missing value](#sub-3-2)

* [3.3. Cleaning data](#sub-3-3)

* [3.4. Feature EDA](#sub-3-4)

[4. DATA PREPROCESSING](#sec-4)

* [4.1. Data Splitting](#sub-4-1)

* [4.2. Scale data using Standard Scaler](#sub-4-2)

[5. TRAINING THE MODEL](#sec-5)

* [5.1. Evaluating Performance](#sub-5-1)

* [5.2. Hyperparameter Tuning](#sub-5-2)

* [5.3. Apply penalty L1 L2 Neural Model](#sub-5-3)

[6. FINAL EVALUATION AND INTERPRETATION OF RESULT](#sec-6)

# 1. **IMPORT** **LIBRARIES**

In [ ]:
# Import the necessary libraries the basic ones
import json
import math
import yfinance as yf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, date

#Import color pallet and theme
import matplotlib.colors
palette_years = sns.color_palette("coolwarm", 8)
palette_seasons = sns.color_palette("Set2", 4)
palette_weekdays = sns.color_palette("coolwarm", 7)
pal= ['#009473', '#00537c', '#b4b4b4', '#da3e21',
          '#ff6347', '#ffa07a', '#ffd700', '#ffdab9',
          '#32cd32', '#3cb371', '#8fbc8f', '#00ced1',
          '#4682b4', '#6495ed', '#4169e1', '#9370db',
          '#800080', '#ff69b4', '#ff1493', '#ff7f50']


# Libraries for time series analysis
from tqdm import tqdm
import statsmodels.api as sm
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

# Libraries for scaling data
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler\

# Libraries for evaluate metrics & Function Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
def metrics(y_test, y_pred):
  mse = mean_squared_error(y_test, y_pred)
  mae = mean_absolute_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)
  print('MSE = ', mse)
  print('RMSE = ', rmse)
  print('MAE = ', mae)
  print('r2 = ', r2)

# Libraries with the models to be trained (Regressor)
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor, AdaBoostRegressor, GradientBoostingRegressor

# Libraries with the models to be trained (Neuronal Regressor)
import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.neural_network import MLPRegressor
from keras import regularizers

from statsmodels.tsa.statespace.sarimax import SARIMAX


# Remove  warnings
import sys
import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")


# 2. **DATA** **LOADING**




---


---


**A brief description of the columns, aiming for better understanding and interpretation of the dataset**


---


1. 'data': Date of observation.
2. 'energy_china_price': Price of electricity in China.
3. 'btc_price': Bitcoin price in USD.
4. 'eur_usd': EUR/USD exchange rate.
5. 'gold_price': Price of gold in USD.
6. 'coal_china_price': Price of coal in China.
7. 's&p500': S&P 500 index.
8. '200w_moving_avg': 200-week moving average.
9. 'avg_block_size': Average size of Bitcoin blocks.
10. 'avg_confirmation_time': Average confirmation time for Bitcoin transactions.
11. 'blocks_size': Size of Bitcoin blocks.
12. 'difficulty_network': Bitcoin network difficulty.
13. 'transaction_volume_usd': Total transaction volume of Bitcoin in USD.
14. 'transaction_volume_btc': Total transaction volume of Bitcoin in BTC.
15. 'transaction_fees_usd_x': Total transaction fees of Bitcoin in USD.
16. 'hash_rate': Bitcoin network hash rate.
17. 'market_capital': Market capitalization of Bitcoin.
18. 'median_confirmation_time': Median confirmation time for Bitcoin transactions.
19. 'miners_revenue': Bitcoin miners' revenue.
20. 'mvrv': Market value to realized value ratio of Bitcoin.
21. 'n_transactions_per_block': Number of transactions per Bitcoin block.
22. 'n_transactions_total': Total number of Bitcoin transactions.
23. 'n_transactions': Total number of Bitcoin transactions.
24. 'n_unique_addresses': Number of unique Bitcoin addresses.
25. 'nvt': Network Value to Transactions (NVT) ratio.
26. 'nvts': Smoothed Network Value to Transactions (NVT) ratio.
27. 'output_volume': Total output volume of Bitcoin transactions.
28. 'total_bitcoins': Total number of existing Bitcoins.
29. 'trade_volume': Bitcoin trading volume.
30. 'transaction_fees_usd_y': Total transaction fees of Bitcoin in USD.
31. 'transaction_fees': Total transaction fees of Bitcoin.
32. 'transactions_per_second': Number of Bitcoin transactions per second.
33. 'utxo_count': Total number of Bitcoin UTXOs (Unspent Transaction Outputs).
34. 'psihologi_cicle': Bitcoin market psychological cycle.


---



---



In [ ]:
# Import json file
def read_json_files(data_bases):

    data_readers = []

    for data_base in data_bases:
        with open(data_base, 'r') as file:
            data_reader = json.load(file)
            data_readers.append(data_reader)

    return data_readers

# Address files
address = [
    '/content/200w-moving-avg-heatmap.json',
    '/content/avg-block-size.json',
    '/content/avg-confirmation-time.json',
    '/content/blocks-size.json',
    '/content/difficulty.json',
    '/content/estimated-transaction-volume-usd.json',
    '/content/estimated-transaction-volume.json',
    '/content/fees-usd-per-transaction.json',
    '/content/hash-rate.json',
    '/content/market-cap.json',
    '/content/median-confirmation-time.json',
    '/content/miners-revenue.json',
    '/content/mvrv.json',
    '/content/n-transactions-per-block.json',
    '/content/n-transactions-total.json',
    '/content/n-transactions.json',
    '/content/n-unique-addresses.json',
    '/content/nvt.json',
    '/content/nvts.json',
    '/content/output-volume.json',
    '/content/total-bitcoins.json',
    '/content/trade-volume.json',
    '/content/transaction-fees-usd.json',
    '/content/transaction-fees.json',
    '/content/transactions-per-second.json',
    '/content/utxo-count.json'
]

# Attributing names to files
data_200_moving,\
data_average_block,\
data_confirmation_time,\
data_size_block,\
data_dificult,\
data_volume_usd,\
data_volume_btc,\
data_fees_usd,\
data_hash,\
data_market,\
data_conf_time,\
data_miners_revenue,\
data_mvrv,\
data_n_transaction_block,\
data_n_transaction_total,\
data_n_transaction,\
data_uniq_addres,\
data_nvt,\
data_nvts,\
data_output_vol,\
data_total_btc,\
data_trade_vol,\
data_fees_usd,\
data_fees_btc,\
data_transaction_per_secon,\
data_utxo_count = read_json_files(address) # Calling the function


In [ ]:
# Import data from the Yahoo Finance

# Import symbols necessary
symbol = ["601991.SS", "YZCAY", "^GSPC", "GC=F", "EURUSD=X", "BTC-USD"]

# Define the time period
data = yf.download(symbol, start="2015-01-14", end="2024-01-20") # end=datetime.now().strftime('%Y-%m-%d') .... currently loading the data

# Save file
data.to_csv('finance_data.csv', index=True)

In [ ]:
# Read CSV file from Yahoo
address_data = pd.read_csv('/content/finance_data.csv')

# Create DataFrame
df_yfinance = pd.DataFrame(address_data)

# 3. **DATA** **CLEANING**

   # 3.1 Undestend Data & Merge DataSet

In [ ]:
# Visualised MATRIX
data_hash

In [ ]:
# Visualised DataFrame Yahoo
df_yfinance.head()

In [ ]:
# Convert matrix format to specified datetime format
def convert_format_matrix(matrix):

    data = pd.DataFrame(matrix[matrix['metric1']])
    data['timestamp'] = pd.to_datetime(data['x'], unit='ms')
    data = data[['timestamp', 'y']]
    data.columns = ['Data', matrix['metric1']]

    return data

In [ ]:
# Create list with json files
list_matrix = [
                data_200_moving,
                data_average_block,
                data_confirmation_time,
                data_size_block,
                data_dificult,
                data_volume_usd,
                data_volume_btc,
                data_fees_usd,
                data_hash,
                data_market,
                data_conf_time,
                data_miners_revenue,
                data_mvrv,
                data_n_transaction_block,
                data_n_transaction_total,
                data_n_transaction,
                data_uniq_addres,
                data_nvt,
                data_nvts,
                data_output_vol,
                data_total_btc,
                data_trade_vol,
                data_fees_usd,
                data_fees_btc,
                data_transaction_per_secon,
                data_utxo_count
                ]

# Create a list comprehension to iterate through the list
list_transform = [convert_format_matrix(data) for data in list_matrix]

In [ ]:
# Colling list comprehension function and apply to all json files
data_200_moving,\
data_average_block,\
data_confirmation_time,\
data_size_block,\
data_dificult,\
data_volume_usd,\
data_volume_btc,\
data_fees_usd,\
data_hash,\
data_market,\
data_conf_time,\
data_miners_revenue,\
data_mvrv,\
data_n_transaction_block,\
data_n_transaction_total,\
data_n_transaction,\
data_uniq_addres,\
data_nvt,\
data_nvts,\
data_output_vol,\
data_total_btc,\
data_trade_vol,\
data_fees_usd,\
data_fees_btc,\
data_transaction_per_secon,\
data_utxo_count = list_transform

In [ ]:
# Chenge format columns Date
def format_column_data(df_list):

# Define a for loop to iterate through the list
  for df in df_list:
    df['Data'] = df['Data'].dt.strftime('%Y-%m-%d')
    df['Data'] = pd.to_datetime(df['Data'], errors='coerce')

# Create a list
df_list = [data_market,
           data_mvrv,
           data_nvt,
           data_nvts,
           data_total_btc,
           data_transaction_per_secon,
           data_utxo_count]

# Colling function
format_column_data(df_list)

In [ ]:
# Create a list with the columns name
columns_yfinance = ['Price', 'Adj Close', 'Adj Close.1', 'Adj Close.2',
                    'Adj Close.3', 'Adj Close.4', 'Adj Close.5']

# Rename Columns
df_yfinance_renamed = df_yfinance[columns_yfinance].copy().rename(columns={'Price': 'Data',
                                                                           'Adj Close': 'energy_china_price',
                                                                           'Adj Close.1': 'btc_price',
                                                                           'Adj Close.2': 'eur_usd',
                                                                           'Adj Close.3': 'gold_price',
                                                                           'Adj Close.4': 'coal_china_price',
                                                                           'Adj Close.5': 's&p500'}).drop(df_yfinance.index[0:2]).reset_index(drop=True)

#List column names
print(df_yfinance_renamed.columns)

In [ ]:
# Change type columns
def convert_data(df_yfinance_renamed):

  df_yfinance_renamed['Data'] = pd.to_datetime(df_yfinance_renamed['Data'],
                                               errors='coerce')

# Define a loop to iterate through columns
  for col in df_yfinance_renamed.columns:
    if df_yfinance_renamed[col].dtype == 'object':
      df_yfinance_renamed[col] = df_yfinance_renamed[col].astype('float64')

  return df_yfinance_renamed

# Colling the function and defined new DFrame Yahoo
df_yfinance_renamed = convert_data(df_yfinance_renamed)

In [ ]:
# Check info DFrame Yahoo
df_yfinance_renamed.info()

In [ ]:
# Check rows DFrame Yahoo
df_yfinance_renamed.shape

In [ ]:
# DataFrame Yahoo renamed
df_yfinance_renamed.head()

In [ ]:
# We merge all the JSON files from the blockchain website
def merge_dataset(df_yfinance_renamed, list_data, how='left', on='Data'):

    data_result = df_yfinance_renamed

# Iterate through
    for data in list_data:
        for column in data.columns:
            if data[column].dtype == 'object' or data[column].dtype == 'bool':
                data[column] = pd.to_datetime(data[column], errors='coerce')

        data_result = pd.merge(data_result, data, how=how, on=on)
    return data_result

# Create list of all files
list_data = [
              data_200_moving,
              data_average_block,
              data_confirmation_time,
              data_size_block,
              data_dificult,
              data_volume_usd,
              data_volume_btc,
              data_fees_usd,
              data_hash,
              data_market,
              data_conf_time,
              data_miners_revenue,
              data_mvrv,
              data_n_transaction_block,
              data_n_transaction_total,
              data_n_transaction,
              data_uniq_addres,
              data_nvt,
              data_nvts,
              data_output_vol,
              data_total_btc,
              data_trade_vol,
              data_fees_usd,
              data_fees_btc,
              data_transaction_per_secon,
              data_utxo_count
              ]

# Calling  function and create DFrame with Bitcoin values
DataFrame_Merged = merge_dataset(df_yfinance_renamed, list_data)


  # 3.2 Check and Input Missing value


In [ ]:
# Size check dataframe
DataFrame_Merged.shape

In [ ]:
# Check Nan values
print(f"Check Nan_Values: \n{DataFrame_Merged.isna().sum()}")

**Notice**


---


We encounter a considerable amount of NaN values in our dataset. The occurrence of these can be attributed to two aspects. Firstly, we observe NaN values on weekends. Secondly, we encounter missing data that does not match specific dates, resulting in the creation of new rows or the merging of data, which consequently contain NaN values


---



In [ ]:
# View the data set
DataFrame_Merged.head()

In [ ]:
# We are writing a function that will help us replace NaN values with the weekly mean where NaN values are present

# Defined function
def weekend_nan_replace_mean_values(df):
  # Iterate through columns
    for col in df.columns:
      # we find the nan values and define the weekend days
        if df[col].isna().any():
            weekend_data = df[df['Data'].dt.dayofweek >= 5]
            # iterate through the rows and replace the nan values with the average of the week
            for index, row in weekend_data.iterrows():
                year = row['Data'].year
                month = row['Data'].month
                week = row['Data'].isocalendar().week
                month_week_data = df[(df['Data'].dt.year == year) & (df['Data'].dt.month == month) & (df['Data'].dt.isocalendar().week == week)]
                month_week_mean = month_week_data[col].mean()
                if pd.isna(row[col]):
                    df.at[index, col] = month_week_mean
    return df

# Call function
DataFrame_Merged = weekend_nan_replace_mean_values(DataFrame_Merged)



In [ ]:
# Check Nan values
print(f"Check Nan_Values: \n{DataFrame_Merged.isna().sum()}")

**Notice**


---


Notice a considerable number of NaN values have been replaced with the weekly mean. The remaining values will be replaced using linear interpolation, which is a good method in our case, as it draws a line between known values, and the NaN values are calculated from these known points


---



In [ ]:
# We replace the NaN values with linear interpolation method
def interpolate_values(df, column):

# We iterate through the values
  for val in column:
    if df[val].isnull().values.any():
       df[val] = df[val].interpolate(method='linear', axis=0)

# Call the function
interpolate_values(DataFrame_Merged, DataFrame_Merged.columns)

# Check Nan values after Interpolate
print(f"Check Nan_Values after Interpolate Values:\n{DataFrame_Merged.isna().sum()}\n")

# Delete Nan values
DataFrame_Merged.dropna(inplace=True)

# Check Nan values before Interpolate
print(f"Check Nan_Values after deleted Nan Values:\n{DataFrame_Merged.isna().sum()}")

**Notice**


---


After applying the interpolation technique, some columns still contain NaN values. This occurred because in the absence of nearby data points, the mentioned technique was unable to draw a line between the available data points and replace the NaN values.
Considering the size of the dataset, I have decided to delete the rows with missing values. This manipulation results in retaining rows adequately sized for training a model.


---



  # 3.3. Cleaning data

In [ ]:
# Create new column Psihologi market cicle
def psihologi_market_cicle(df):

# Iterate through the Date column and define market psychology
  for idx, row in df.iterrows():
    if row['Data'] >= pd.Timestamp('2017-01-01') and row['Data'] <= pd.Timestamp('2017-10-15'):
      df.loc[idx, 'psihologi_cicle'] = 'optimism'

    elif row['Data'] >= pd.Timestamp('2017-10-16') and row['Data'] <= pd.Timestamp('2017-11-20'):
      df.loc[idx, 'psihologi_cicle'] = 'belief'

    elif row['Data'] >= pd.Timestamp('2017-11-21') and row['Data'] <= pd.Timestamp('2017-12-04'):
      df.loc[idx, 'psihologi_cicle'] = 'thrill'

    elif row['Data'] >= pd.Timestamp('2017-12-05') and row['Data'] <= pd.Timestamp('2018-01-07'):
      df.loc[idx, 'psihologi_cicle'] = 'euphoria'

    elif row['Data'] >= pd.Timestamp('2018-01-08') and row['Data'] <= pd.Timestamp('2018-01-29'):
      df.loc[idx, 'psihologi_cicle'] = 'complacency'

    elif row['Data'] >= pd.Timestamp('2018-01-30') and row['Data'] <= pd.Timestamp('2018-04-29'):
      df.loc[idx, 'psihologi_cicle'] = 'anxiety'

    elif row['Data'] >= pd.Timestamp('2018-04-30') and row['Data'] <= pd.Timestamp('2018-11-05'):
      df.loc[idx, 'psihologi_cicle'] = 'denial'

    elif row['Data'] >= pd.Timestamp('2018-11-06') and row['Data'] <= pd.Timestamp('2018-11-19'):
      df.loc[idx, 'psihologi_cicle'] = 'panic'

    elif row['Data'] >= pd.Timestamp('2018-11-20') and row['Data'] <= pd.Timestamp('2019-01-21'):
      df.loc[idx, 'psihologi_cicle'] = 'anger'

    elif row['Data'] >= pd.Timestamp('2019-01-22') and row['Data'] <= pd.Timestamp('2020-02-17'):
      df.loc[idx, 'psihologi_cicle'] = 'depresion'

    elif row['Data'] >= pd.Timestamp('2020-02-18') and row['Data'] <= pd.Timestamp('2020-04-09'):
      df.loc[idx, 'psihologi_cicle'] = 'disbelief'

    elif row['Data'] >= pd.Timestamp('2020-04-10') and row['Data'] <= pd.Timestamp('2020-10-05'):
      df.loc[idx, 'psihologi_cicle'] = 'hope'

    elif row['Data'] >= pd.Timestamp('2020-10-06') and row['Data'] <= pd.Timestamp('2020-12-07'):
      df.loc[idx, 'psihologi_cicle'] = 'optimism'

    elif row['Data'] >= pd.Timestamp('2020-12-08') and row['Data'] <= pd.Timestamp('2021-01-18'):
      df.loc[idx, 'psihologi_cicle'] = 'belief'

    elif row['Data'] >= pd.Timestamp('2021-01-19') and row['Data'] <= pd.Timestamp('2021-02-15'):
      df.loc[idx, 'psihologi_cicle'] = 'thrill'

    elif row['Data'] >= pd.Timestamp('2021-02-15') and row['Data'] <= pd.Timestamp('2021-05-03'):
      df.loc[idx, 'psihologi_cicle'] = 'euphoria'

    elif row['Data'] >= pd.Timestamp('2021-05-04') and row['Data'] <= pd.Timestamp('2021-12-20'):
      df.loc[idx, 'psihologi_cicle'] = 'complacency'

    elif row['Data'] >= pd.Timestamp('2021-12-21') and row['Data'] <= pd.Timestamp('2022-03-28'):
      df.loc[idx, 'psihologi_cicle'] = 'anxiety'

    elif row['Data'] >= pd.Timestamp('2022-03-29') and row['Data'] <= pd.Timestamp('2022-05-23'):
      df.loc[idx, 'psihologi_cicle'] = 'denial'

    elif row['Data'] >= pd.Timestamp('2022-05-24') and row['Data'] <= pd.Timestamp('2022-07-11'):
      df.loc[idx, 'psihologi_cicle'] = 'panic'

    elif row['Data'] >= pd.Timestamp('2022 -07-12') and row['Data'] <= pd.Timestamp('2023-01-02'):
      df.loc[idx, 'psihologi_cicle'] = 'anger'

    elif row['Data'] >= pd.Timestamp('2023-01-03') and row['Data'] <= pd.Timestamp('2023-08-28'):
      df.loc[idx, 'psihologi_cicle'] = 'depresion'

    elif row['Data'] >= pd.Timestamp('2023-08-29') and row['Data'] <= pd.Timestamp('2023-12-01'):
      df.loc[idx, 'psihologi_cicle'] = 'disbelief'

    else:
      df.loc[idx, 'psihologi_cicle'] = 'hope' # This condition is valid until January 2024.
                                              # Before processing the data for future months, this condition needs to be revised based on a technical analysis of the chart

  return df

# Call function
DataFrame_Merged = psihologi_market_cicle(DataFrame_Merged)


In [ ]:
"""We resize the dataset from 2017 to the present to have a more consistent trend of BTC"""

# Define a new dataframe with values from 2017 to the present
data = DataFrame_Merged[(DataFrame_Merged['Data'] >= '2017-01-01') &\
                        (DataFrame_Merged['Data'] <= DataFrame_Merged['Data'].max())]

In [ ]:
# Rename columns for clarity
data = data.rename(columns={'Data': 'data',
                            '200w-moving-avg-heatmap': '200w_moving_avg',
                            'avg-block-size': 'avg_block_size',
                            'avg-confirmation-time': 'avg_confirmation_time',
                            'blocks-size': 'blocks_size',
                            'difficulty': 'difficulty_network',
                            'estimated-transaction-volume-usd': 'transaction_volume_usd',
                            'estimated-transaction-volume': 'transaction_volume_btc',
                            'transaction-fees-usd_x': 'transaction_fees_usd_x',
                            'hash-rate': 'hash_rate',
                            'market-cap': 'market_capital',
                            'median-confirmation-time': 'median_confirmation_time',
                            'miners-revenue': 'miners_revenue',
                            'n-transactions-per-block': 'n_transactions_per_block',
                            'n-transactions-total': 'n_transactions_total',
                            'n-transactions': 'n_transactions',
                            'n-unique-addresses': 'n_unique_addresses',
                            'output-volume': 'output_volume',
                            'total-bitcoins': 'total_bitcoins',
                            'trade-volume': 'trade_volume',
                            'transaction-fees-usd_y': 'transaction_fees_usd_y',
                            'transaction-fees': 'transaction_fees',
                            'transactions-per-second': 'transactions_per_second',
                            'utxo-count': 'utxo_count'
                            })

#List column names
print(data.columns)

In [ ]:
# Move the target column to the last position
colum = 'btc_price'
column_replace = data.pop(colum)
data[colum] = column_replace


In [ ]:
"""This piece of code can be adjusted according to our preference, specifying the number of days we want to make predictions for"""

# Define a new target column with the date back
data['btc_shifted'] = data['btc_price'].shift(-5)

# Remove th Nan values from the last row
data = data.iloc[:-5]

In [ ]:
# Visualise the last 5 rows
data.tail()

   # 3.4. Feature EDA

**Notice**


---


***At this stage, we will visualize and analyze the data using various time series manipulation methods to select the most relevant columns for model training and achieve a more accurate prediction***


---



In [ ]:
# Description of the data in the dataframe
data.describe().T

**Notice**


---


After analyzing the dataset description, I noticed that the largest anomalies in the data are in the 'eur-usd' column and the 'n-transactions_total' column. In the next analyses, I will scrutinize these columns more closely to decide whether to eliminate them or not.


---



In [ ]:
# Visualize correlation columns
plt.figure(figsize=(28, 12))
sns.heatmap(data.corr(), annot=True)
plt.title('Correlation Columns')
plt.show()

**Notice**


---


This graph demonstrates a fairly strong correlation among certain columns that could be selected for further processing. The columns that draw my attention at this stage are:





*   gold_price
*   s&p500
*   coal_china_price
*   200w_moving_avg
*   blocks_size
*   difficulty_network
*   transaction_volume_usd
*   hash_rate
*   market_capital
*   miners_revenue
*   n_transactions_total
*   n_unique_addresses
*   utxo_count
*   total_bitcoins


Some of the columns have a fairly strong correlation, which could be a negative point for my model, increasing the chances of overfitting. These columns were initially included in the list, but during the analysis, I will decide whether to keep or remove them.


---



In [ ]:
# Check info DSet
data.info()

In [ ]:
"""We visualize the actual BTC data using a plot"""

# Set dimension plot
plt.figure(figsize=(12, 6))
plt.style.use('seaborn-darkgrid')

data['btc_price'].plot.line(x=data['data'], color='royalblue', linewidth=2, label='Price Bitcoin')

# Add title
plt.title('Bitcoin Price Time Series', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Price (USD)', fontsize=14)

# add legend
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

# Display plot
plt.tight_layout()
plt.show()


In [ ]:
"""This graph was created based on a meticulous analysis of the BTC price trend,
leading to conclusions regarding trader buying and selling psychology. Additionally,
it allows us to determine the current market cycle."""

# Set dimension plot
plt.figure(figsize=(15,6))
# Set style
plt.style.use('seaborn')
# Create plot
ax = sns.scatterplot(data=data, x='data', y='btc_price', hue='psihologi_cicle', palette=pal)

# add title
ax.set_title('Price Bitcoin in correlation Psihologi Cicle Human', fontsize=20)
ax.set_xlabel('Data Represented Cycle Psihologic', fontsize=14)
ax.set_ylabel('Price')

# Display plot
plt.legend()
plt.show()


In [ ]:
"""This piece of code will calculate the behavior of BTC price day by day,
season by season, and year by year, highlighting the days, seasons,
and years when BTC experienced the most significant increases and decreases"""



def calculate_average_btc_price(df):
    # Extract the day of the week and the season from the DataFrame date
    week_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df['Day_of_Week'] = pd.Categorical(df['Data'].dt.day_name(), categories=week_days, ordered=True)

    year_temp = ['Spring', 'Summer', 'Autumn', 'Winter']
    df['Season'] = pd.Categorical(df['Data'].dt.month % 12 // 3, categories=[0, 1, 2, 3], ordered=True)
    df['Season'] = df['Season'].map(dict(zip(range(4), year_temp)))

    # Calculate mean price btc for every day for every season
    avg_prices_day_of_week = df.groupby('Day_of_Week')['btc_price'].mean().reset_index()
    avg_prices_season = df.groupby('Season')['btc_price'].mean().reset_index()

    # Add column year
    df['Year'] = df['Data'].dt.year

    # Calculate mean price btc for every year
    avg_prices_year = df.groupby('Year')['btc_price'].mean().reset_index()

    avg_prices_day_of_week.columns = ['Day', 'Average_Price_DayofWeek']
    avg_prices_season.columns = ['Season', 'Average_Price_Season']
    avg_prices_year.columns = ['Year', 'Average_Price_Year']

    return avg_prices_day_of_week, avg_prices_season, avg_prices_year

# Call function
avg_prices_day_of_week, avg_prices_season, avg_prices_years = calculate_average_btc_price(DataFrame_Merged)


In [ ]:
# We scale the dates on a scale from 0 to 1% to make them easier to interpret

avg_prices_day_of_week['Procentge_Price_Days'] = round(round(avg_prices_day_of_week['Average_Price_DayofWeek'] * \
                                                             100 / avg_prices_day_of_week['Average_Price_DayofWeek'].sum(), 4)/100, 5)

avg_prices_season['Procentge_Price_Season'] = round(round(avg_prices_season['Average_Price_Season'] * \
                                                          100 / avg_prices_season['Average_Price_Season'].sum(), 4)/100, 5)

avg_prices_years['Procentge_Price_Year'] = round(round(avg_prices_years['Average_Price_Year'] * \
                                                          100 / avg_prices_years['Average_Price_Year'].sum(), 4)/100, 5)

In [ ]:
# We visualize the temporary DataFrame with the day of the week column

avg_prices_day_of_week

In [ ]:
fig = plt.figure(figsize = (12, 4))


# create barplot using seaborn where x is products in the category, y is the current iteration of the cl list
ax = sns.barplot(data=avg_prices_day_of_week, x='Procentge_Price_Days', y='Day',
                palette=palette_weekdays, linestyle="-", linewidth=1, edgecolor="black")
plt.xticks(size=13, color='black')
plt.yticks(size=13, color='black')
plt.xlim(0.142, 0.144)
plt.xticks([])
plt.title("BTC Price Fluctuations Across Weekdays <Scale>(0% Min - 1% Max)", pad=15, size=16)
plt.xlabel('Average Procentage Price of Bitcoin')
plt.ylabel('Day of the Week')

# add annotations to the barplot
for i, v in enumerate(avg_prices_day_of_week['Procentge_Price_Days']):
    ax.text(v + 0.0001, i, (str(v)+"%"), color='black', fontsize=16)

for j in ['right', 'top', 'left', 'bottom']:
    ax.spines[j].set_visible(False)

plt.show()


In [ ]:
# We visualize the temporary DataFrame with the season column

avg_prices_season

In [ ]:
fig = plt.figure(figsize = (12, 4))


# create barplot using seaborn where x is products in the category, y is the current iteration of the cl list
ax = sns.barplot(data=avg_prices_season, x='Procentge_Price_Season', y='Season',
                palette=palette_seasons, linestyle="-", linewidth=1, edgecolor="black")
plt.xticks(size=13, color='black')
plt.yticks(size=13, color='black')
plt.xlim(0.2, 0.3)
plt.xticks([])
plt.title("BTC Price Fluctuations Across Season <Scale>(0% Min - 1% Max)", pad=15, size=16)
plt.xlabel('Average Procentage Price of Bitcoin')
plt.ylabel('Season of the Year')

# add annotations to the barplot
for i, v in enumerate(avg_prices_season['Procentge_Price_Season']):
    ax.text(v + 0.0001, i, (str(v)+"%"), color='black', fontsize=16)

for j in ['right', 'top', 'left', 'bottom']:
    ax.spines[j].set_visible(False)

plt.show()


In [ ]:
# We visualize the temporary DataFrame with the year column

avg_prices_years

In [ ]:
# We remove the first row containing a NaN value
avg_prices_year = avg_prices_years.loc[1:]

In [ ]:
fig = plt.figure(figsize=(10, 5))

# Create plot using Seaborn
sns.lineplot(data=avg_prices_year, x='Year', y='Procentge_Price_Year', color='skyblue', linewidth=2.5)

# Add annotation %
for i, row in avg_prices_year.iterrows():
    plt.scatter(row['Year'], row['Procentge_Price_Year'], color='skyblue', s=100)
    plt.text(row['Year'], row['Procentge_Price_Year'], f"{row['Procentge_Price_Year']*100:.2f}%", color='black', fontsize=12, va='bottom')

# Configure title and etickets axes
plt.title("BTC Price Fluctuations Across Years <Scale>(0% Min - 100% Max)", pad=15, size=16)
plt.xlabel('Year')
plt.yticks([])

# Add margin axes
for j in ['right', 'top', 'left', 'bottom']:
    plt.gca().spines[j].set_visible(False)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


**Notice**


---


* From the analysis of these three graphs, it can be observed that the price of BTC shows a more pronounced increase on Wednesdays and Fridays, with a price decrease during the weekends.
* The most favorable seasons for BTC are spring and summer, while the least favorable season is autumn.
* If we analyze the BTC price over the years, we observe a consistent upward trend. The price shows an annual increase ranging from 2% to 27%, with the most notable years being 2021 with 26.48% and 2023 with 24.48% growth. Considering the significant growth observed in 2023, we might anticipate a price decrease in 2024


---



In [ ]:

def show_plot_correlation_columns(df, col, threshold=0.20):
    # Set style graph
    sns.set_context("paper")
    sns.set_style("whitegrid")

    # DDefined colors
    palette = sns.color_palette("husl", n_colors=2)



    # We iterate through each column and create a separate plot for the columns that have a correlation greater than or equal to a certain threshold
    for column in df.columns:
        if column != col and df[column].dtype != 'datetime64[ns]' and df[column].dtype != 'object' and column != 'btc_shifted':
            correlation = df[column].corr(df[col])
            if correlation >= threshold:
                plt.figure(figsize=(12, 6))
                ax1 = sns.lineplot(data=df, x='data', y=col, color=palette[0], label=col)

                # Creating a secondary axis to plot the second column
                ax2 = ax1.twinx()
                sns.lineplot(data=df, x='data', y=column, color=palette[1], label=column, ax=ax2)

                # Add title and etickets for axes x and y
                plt.title(f"Correlation between {col} and {column} (Correlation: {correlation:.2f})", fontsize=16)
                plt.xlabel('Date', fontsize=14)
                ax1.set_ylabel(f'{col} Price', fontsize=14)
                ax2.set_ylabel(f'{column} Price', fontsize=14)

                # Config legend
                ax1.legend(loc='upper left', fontsize=12)
                ax2.legend(loc='upper right', fontsize=12)
                plt.grid(alpha=0.3)

                # Display plot
                plt.tight_layout()
                plt.show()

# Call function
show_plot_correlation_columns(data, 'btc_price')


**Notice**


---


After analyzing this graph, we have concluded that we will keep the selected columns mentioned above. Only two columns show a very strong correlation: 200w_moving_average and market_capital. The fate of these columns will be decided during the analysis.

I have noticed that these columns have economic relevance and significance in terms of trading volume , a constant correlation between prices



---



In [ ]:
# Decomposition of each column to visualize and analyze seasonal trends and residuals
def insight_data(data_series):

# Defined variable with columns
    select_columns = [data for data in data_series.columns if data != 'psihologi_cicle' and data != 'btc_shifted' and data != 'data']

# Iterate through each column
    for column in select_columns:
        x = data_series[column]

# Decomposition of columns into trends
        ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
        trend = ss_decomposition.trend
        seasonal = ss_decomposition.seasonal
        residual = ss_decomposition.resid

# visualised result
        fig, ax = plt.subplots(nrows=4, ncols=1, figsize=(10, 6))
        ax[0].plot(x, color='green', label='Original')
        ax[0].legend(loc='upper right')
        ax[0].set_title(column)

        ax[1].plot(trend, color='gray', label='Trend')
        ax[1].legend(loc='upper right')
        ax[1].set_title(column)

        ax[2].plot(seasonal, color='lightblue', label='Seasonal')
        ax[2].legend(loc='upper right')
        ax[2].set_title(column)

        ax[3].plot(residual, color='purple', label='Residual')
        ax[3].legend(loc='upper right')
        ax[3].set_title(column)

# Setting grid lines transparency for better readability
        for x in ax:
          x.grid(alpha=0.3)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.5, wspace=1)
        plt.show()


# Calling function
insight_data(data)


**Notice**


---


* Making a conclusion about the selected columns, we observe that there is a seasonal or cyclical trend that would influence the price of BTC.
* Upon examining the residuals, we notice a significant correlation between the columns.
* We observe that the time evolution of the columns does not represent a temporal correlation, but rather the opposite.


---



**Notice**


---


We notice that the mentioned columns, their trends are similar to the price of BTC, and volatility or variability similar to that of the Bitcoin price are more relevant for prediction. Similarly, we also see similar cyclic patterns.


---



In [ ]:
# Set the global style to 'ticks'
sns.set_style('ticks')

# Define the size of the plot
fig, ax = plt.subplots(nrows=11, ncols=3, figsize=(15, 30))

# Flatten the 2D array of axes to iterate over each axis
ax = ax.flatten()

def visualize_addfuller_results(series, title, ax):
    # Compute the Augmented Dickey-Fuller test
    result = adfuller(series)
    significance_level = 0.05
    adf_stat = result[0]
    p_val = result[1]

    # Extract critical values
    crit_val_1 = result[4]['1%']
    crit_val_5 = result[4]['5%']
    crit_val_10 = result[4]['10%']

    # Determine the color for stationary series
    if (p_val < significance_level) & (adf_stat < crit_val_1):
        linecolor = 'forestgreen'
    elif (p_val < significance_level) & (adf_stat < crit_val_5):
        linecolor = 'orange'
    elif (p_val < significance_level) & (adf_stat < crit_val_10):
        linecolor = 'red'
    else:
        linecolor = 'purple'

    # Display a lineplot
    sns.lineplot(x=data.index, y=series, ax=ax, color=linecolor)

    ax.set_title(f'ADF Statistic {adf_stat:0.3f}, p-value: {p_val:0.3f}\nCritical Values 1%: {crit_val_1:0.3f}, 5%: {crit_val_5:0.3f}, 10%: {crit_val_10:0.3f}',
                 fontsize=12)
    ax.set_ylabel(ylabel=title, fontsize=14)
    ax.grid(alpha=0.3)

columns_all = [col for col in data.columns if col != 'psihologi_cicle' and col != 'btc_shifted' and col != 'data']

# Iterate through each column and apply the visualization function
for i, column in enumerate(columns_all):
    visualize_addfuller_results(data[column], column, ax[i])

# Remove the last subplot
fig.delaxes(ax[-1])

# Show the plot
plt.tight_layout()
plt.subplots_adjust(hspace=1.5)
plt.show()


**Notice**


---


Through this graph, we can observe which series are stationary and which are not. Columns that have a stationary trend are better modeled and lead to more accurate predictions. For the columns that we do not find stationary, we will later apply manipulative methods to transform them into stationary ones.

From the selected columns, we have one stationary column, "transaction_volume_usd" ... The rest of the columns are non-stationary.

Later, we will apply some manipulative methods to make them stationary for our model


---



In [ ]:
data = data.dropna()

In [ ]:
# We create a function to compare the distributions of transformed data for each column
def compar_date_transformed_distribution(data_series):

    # Defined variable with columns
    columns = [data for data in data_series.columns if data != 'psihologi_cicle']
    # Create Dframe
    data_series_diff = pd.DataFrame()

# Iterate through each column, calculate the difference between consecutive data points
    for col in columns:
        fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20, 4))

        series_diff = np.diff(data_series[col])

        data_series_diff[f"{col}_diff_1"] = np.append([0], series_diff)

# Calling function and transformed series
        visualize_addfuller_results(data_series_diff[f"{col}_diff_1"], f"Transformed {col} Diff", ax[0])
        sns.distplot(data_series_diff[f"{col}_diff_1"], ax=ax[1])
        ax[1].grid(alpha=0.3)
        plt.show()

# Return DFrame
    return data_series_diff

# Calling function and we create a temporary dataframe to visualize the result
data_new_diff = compar_date_transformed_distribution(data)


**Notice**


---


By this plot, we see that among the selected columns, the Gold and Transaction_volume_usd columns have a more uniform distribution, while the rest of the columns have a non-uniform distribution

**Summary of Data Analysis Stage**



---


I have made a brief summary of this data analysis stage, and through the table represented below, it can be seen that not all selected columns from the correlation graph exhibit a uniform distribution or a perfect correlation. However, following this analysis, we have decided that columns accumulating at least 2 positives will be selected for the next preprocessing stage.

Thus, we conclude that from our list, we will delete 1 column, namely: Block_size.



---



**Summary Table of Column Selection**

| Column              | Decomposition   | Rolling_Statistic | Dickey_Fuller | Distribution |
|---------------------|-----------------|-------------------|---------------|--------------|
| gold_price          |        +         |         +          |       -        |      -        |
| s&p500              |        +         |          +         |        -       |       -       |
| coal_china_price    |        +         |          +         |        -       |       -       |
| 200w_moving_avg     |        +         |         +          |       -        |       -       |
| blocks_size         |        +         |          -          |        -       |       -       |
| difficulty_network  |       +          |         +          |       -        |       -       |
| transaction_volume_usd|      +         |         -          |       -        |       +       |
| hash_rate           |        +         |         +          |       -        |       -       |
| market_capital      |        +         |         +          |        +       |       -       |
| miners_revenue      |        +         |         +          |       +        |       -       |
| n_transactions_total|        +         |         -          |       +        |       -       |
| n_unique_addresses  |        +         |         +          |       -        |       -       |
| utxo_count          |        +         |         +          |        +       |       -       |
| total_bitcoins      |        +         |         +          |       +        |       -       |



---



---



# **4. DATA PRE-PROCESSING**

**Notice**


---


* At this stage, we will create a new dataframe with the selected columns.
* We will create 5 new columns with the average percentage of price by day, season, and year.
* Then, for the non-stationary columns, we will apply the logarithmic method and the stationarity decomposition method, and finally, we will analyze the results of these methods on the trends in the columns.


---



In [ ]:
# We define a new dataframe from the analyzed results of the above EDA
data_preprocessing =data[['data',
                          'btc_shifted',
                          '200w_moving_avg',
                          'hash_rate',
                          'market_capital',
                          'miners_revenue',
                          's&p500',
                          'gold_price',
                          'coal_china_price',
                          'difficulty_network',
                          'n_unique_addresses',
                          'utxo_count',
                          'transaction_volume_usd',
                          'n_transactions_total']]



In [ ]:

def calculate_average_btc_price(df):
    # Extract the day of the week and the season from the DataFrame date
    week_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df['day_of_week'] = pd.Categorical(df['data'].dt.day_name(), categories=week_days, ordered=True)

    year_temp = ['Spring', 'Summer', 'Autumn', 'Winter']
    df['season'] = pd.Categorical(df['data'].dt.month % 12 // 3, categories=[0, 1, 2, 3], ordered=True)
    df['season'] = df['season'].map(dict(zip(range(4), year_temp)))

    # Calculate the average Bitcoin price for each day of the week
    avg_prices_day_of_week = df.groupby('day_of_week')['btc_shifted'].mean()

    # Calculate the average Bitcoin price for each day of the season
    avg_prices_season = df.groupby('season')['btc_shifted'].mean()

    # Calculate the average Bitcoin price for each day of the year
    avg_prices_year = df.groupby(df['data'].dt.year)['btc_shifted'].mean()

    # Add new columns
    df['avg_%_day'] = df['day_of_week'].map(avg_prices_day_of_week)
    df['avg_%_season'] = df['season'].map(avg_prices_season)
    df['avg_%_year'] = df['data'].dt.year.map(avg_prices_year)

    return df

# Function call
data_preprocessing = calculate_average_btc_price(data_preprocessing)


In [ ]:
data_preprocessing.info()



---


**The newly created columns are categorical type. We will apply a method to transform them into float.**


---



In [ ]:
# We create a function that iterates through columns and replaces the data type of categorical columns with float.

def transform_type_column(data_preprocessing):

    data_preprocessing.set_index('data', inplace=True)

    for col in data_preprocessing.columns:
        if data_preprocessing[col].dtype.name == 'category' and col not in ['day_of_week', 'season']:

            data_preprocessing[col] = data_preprocessing[col].astype('float64')
    return data_preprocessing

# Function call
transformed_data = transform_type_column(data_preprocessing)


In [ ]:
# We round the values of the new columns to four decimal places for easier interpretation

data_preprocessing['avg_%_day'] = round(data_preprocessing['avg_%_day'] * \
                                                             100 / data_preprocessing['avg_%_day'].sum(), 4)

data_preprocessing['avg_%_season'] = round(data_preprocessing['avg_%_season'] * \
                                                          100 / data_preprocessing['avg_%_season'].sum(), 4)

data_preprocessing['avg_%_year'] = round(data_preprocessing['avg_%_year'] * \
                                                          100 / data_preprocessing['avg_%_year'].sum(), 4)

In [ ]:
# View Dframe
data_preprocessing.head().T

In [ ]:
# Apply method map for the columns that contain values dtype string

data_preprocessing['day_of_week'] = data_preprocessing['day_of_week'].map({'Sunday': 1,
                                                                           'Monday': 2,
                                                                           'Tuesday': 3,
                                                                           'Wednesday': 4,
                                                                           'Thursday': 5,
                                                                           'Friday': 6,
                                                                           'Saturday': 7})

data_preprocessing['season'] = data_preprocessing['season'].map({'Spring': 1,
                                                                 'Summer': 2,
                                                                 'Autumn': 3,
                                                                 'Winter': 4})


# Transform new values to float
data_preprocessing['day_of_week'] = data_preprocessing['day_of_week'].astype('float64')
data_preprocessing['season'] = data_preprocessing['season'].astype('float64')

In [ ]:
# We apply the logarithmic method to each column to observe whether each series has a stationary or non-stationary trend
series_log_1 = np.log(data_preprocessing['btc_shifted'])
series_log_2 = np.log(data_preprocessing['200w_moving_avg'])
series_log_3 = np.log(data_preprocessing['hash_rate'])
series_log_4 = np.log(data_preprocessing['market_capital'])
series_log_5 = np.log(data_preprocessing['miners_revenue'])
series_log_6 = np.log(data_preprocessing['s&p500'])
series_log_7 = np.log(data_preprocessing['gold_price'])
series_log_8 = np.log(data_preprocessing['coal_china_price'])
series_log_9 = np.log(data_preprocessing['n_transactions_total'])
series_log_10 = np.log(data_preprocessing['difficulty_network'])
series_log_11 = np.log(data_preprocessing['n_unique_addresses'])
series_log_12 = np.log(data_preprocessing['utxo_count'])
series_log_13 = np.log(data_preprocessing['transaction_volume_usd'])
series_log_14 = np.log(data_preprocessing['day_of_week'])
series_log_15 = np.log(data_preprocessing['season'])
series_log_16 = np.log(data_preprocessing['avg_%_day'])
series_log_17 = np.log(data_preprocessing['avg_%_season'])
series_log_18 = np.log(data_preprocessing['avg_%_year'])

# Defined list with all Series
list_series_exponential = [series_log_1,
                           series_log_2,
                           series_log_3,
                           series_log_4,
                           series_log_5,
                           series_log_6,
                           series_log_7,
                           series_log_8,
                           series_log_9,
                           series_log_10,
                           series_log_11,
                           series_log_12,
                           series_log_13,
                           series_log_14,
                           series_log_15,
                           series_log_16,
                           series_log_17,
                           series_log_18]


In [ ]:
# We apply the second method of stationary decomposition to transform a series into a stationary trend. For each column

x = data_preprocessing['btc_shifted']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_1 = series_log_1 - seasonal_component

x = data_preprocessing['200w_moving_avg']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_2 = series_log_2 - seasonal_component

x = data_preprocessing['hash_rate']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_3 = series_log_3 - seasonal_component

x = data_preprocessing['market_capital']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_4 = series_log_4 - seasonal_component

x = data_preprocessing['miners_revenue']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_5 = series_log_5 - seasonal_component

x = data_preprocessing['s&p500']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_6 = series_log_6 - seasonal_component

x = data_preprocessing['gold_price']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_7 = series_log_7 - seasonal_component

x = data_preprocessing['coal_china_price']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_8 = series_log_8 - seasonal_component

x = data_preprocessing['n_transactions_total']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_9 = series_log_9 - seasonal_component

x = data_preprocessing['difficulty_network']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_10 = series_log_10 - seasonal_component

x = data_preprocessing['n_unique_addresses']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_11 = series_log_11 - seasonal_component

x = data_preprocessing['utxo_count']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_12 = series_log_12 - seasonal_component

x = data_preprocessing['transaction_volume_usd']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_13 = series_log_13 - seasonal_component

x = data_preprocessing['day_of_week']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_14 = series_log_14 - seasonal_component

x = data_preprocessing['season']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_15 = series_log_15 - seasonal_component

x = data_preprocessing['avg_%_day']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_16 = series_log_16 - seasonal_component

x = data_preprocessing['avg_%_season']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_17 = series_log_17 - seasonal_component

x = data_preprocessing['avg_%_year']
ss_decomposition = seasonal_decompose(x=x, model='additive', period=365)
seasonal_component = ss_decomposition.seasonal
constant_values_18 = series_log_18 - seasonal_component

In [ ]:
# Defined list with Stationary Series
column_constant_values = [constant_values_1,
                          constant_values_2,
                          constant_values_3,
                          constant_values_4,
                          constant_values_5,
                          constant_values_6,
                          constant_values_7,
                          constant_values_8,
                          constant_values_9,
                          constant_values_10,
                          constant_values_11,
                          constant_values_12,
                          constant_values_13,
                          constant_values_14,
                          constant_values_15,
                          constant_values_16,
                          constant_values_17,
                          constant_values_18]

# Iterate list
for col in column_constant_values:

# Show image for every Series
    plt.figure(figsize=(10, 2))
    plt.style.use('ggplot')
    plt.plot(col, color='lightsteelblue', linewidth=1, label=f'{col} Stationary') # color = 'dimgray'
    plt.title(f'{col.name} Display Stationary Trend')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.show()



**Notice**


---


After applying the logarithmic method and the stationary decomposition method, we observe that all columns exhibit a stationary trend.


---



In [ ]:
# Analyzing the autocorrelation and normal distribution of values in each column of the dataset

# Defined variable with columns
column_names = data_preprocessing.columns

# Enumerate  every series from list
for i, column in enumerate(column_constant_values):
    fig, ax = plt.subplots(ncols=2, figsize=(15, 3))

# We visualize a plot to analyze the correlation between a time series and its lagged versions
    sm.graphics.tsa.plot_acf(column, lags=60, ax=ax[0])
    ax[0].set_title(f'ACF for {column_names[i]}')
    ax[0].set_ylim(-1.5, 1.5)

# Compare distribution values series with distribution normal
    stats.probplot(column, dist="norm", plot=ax[1])
    ax[1].set_title(f'Q-Q Plot for {column_names[i]}')

    plt.tight_layout()
    plt.show()



In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
plot_pacf(data_preprocessing['btc_shifted'], lags=20);

**Notice**


---


Through these 2 plots, we can analyze the distribution of each column to see if it approximates the normal distribution or not using the Q-Q plot. Additionally, the ACF plot will assist us in defining the parameters
�
p and
�
q to make predictions through the ARIMA or SARIMAX model.


---



In [ ]:
# We concatenate all series into a DFrame
def merge_series_stationary(series_list):

# Define temporarily DataFrame
    df_series = pd.DataFrame()

    # Enumerate each series an merged
    for idx, series in enumerate(series_list, start=1):
        col_name = f'series_{idx}'
        df_series[col_name] = series

# return DFrame
    return df_series

# Result DFrame for Model
df_model_ready = merge_series_stationary(list_series_exponential)

In [ ]:
# Rename each column for clarity
df_model_ready.rename(columns= {'series_1': 'bitcoin_shifted',
                                'series_2': '200w_moving_average',
                                'series_3': 'hash_rate',
                                'series_4': 'market_capital',
                                'series_5': 'miners_revenue',
                                'series_6': 's&p500',
                                'series_7': 'gold',
                                'series_8': 'coal_china',
                                'series_9': 'transaction_total_market',
                                'series_10': 'difficult_network',
                                'series_11': 'unique_wallet_address',
                                'series_12': 'utxo_count',
                                'series_13': 'transaction_volume_usd',
                                'series_14': 'day_of_week',
                                'series_15': 'season',
                                'series_16': 'avg_%_day',
                                'series_17': 'avg_%_season',
                                'series_18': 'avg_%_year'}, inplace=True)

In [ ]:
# Visualised first 5 rows
df_model_ready.head().T

In [ ]:
cor = df_model_ready.corr()
plt.figure(figsize=(14, 6))
sns.heatmap(cor,annot=True)
plt.show()

**Notice**


---


After this data preprocessing step, we summarize a new dataframe with the most relevant columns to the Bitcoin price, which have a correlation ranging between 50% and 94%, except for the newly created columns, which exhibit a relatively weak correlation, but we decide to keep them. We can observe that the new df columns correlate quite well with the column  avg_%_year


---



   # 4.2 Data Splitting

In [ ]:
# We split the dataset into separate training and testing datasets

last_month = df_model_ready.index.max() - pd.DateOffset(days=120)
features = df_model_ready.drop(columns='bitcoin_shifted').columns

X_train = df_model_ready.loc[:last_month, features]
X_test = df_model_ready.loc[last_month:, features]

y_train = df_model_ready.loc[:last_month, 'bitcoin_shifted']
y_test = df_model_ready.loc[last_month:, 'bitcoin_shifted']

In [ ]:
# Verified dimension
print(f"X_train: {X_train.shape}\n")
print(f"X_test: {X_test.shape}\n")
print(f'y_train: {y_train.shape}\n')
print(f"y_test: {y_test.shape}")

#   4.2. Scale data using Standard Scaler

**Notice**


---


---



At the data splitting stage, we will define two variables, X_test_scaled and X_train_scaled, which will contain the scaled data, as we have algorithms that train better with scaled data, such as:


---


* LinearRegression()
* KNeighborsRegressor()
* MLPRegressor (neural network)
* Sequential (neural network)



and algorithms that do not require scaled data:

* RandomForestRegressor()
* ExtraTreesRegressor()
* XGBRegressor()
* LGBMRegressor()
* AdaBoostRegressor()
* GradientBoostingRegressor()
* DecisionTreeRegressor()
* Sarimax


---


However, during this stage, we will also test with both scaled and unscaled data.


---



---



In [ ]:
# Data scaling
std = StandardScaler()

# Defined variables scaled X_train, X_test
X_train_scaled = std.fit_transform(X_train)
X_test_scaled = std.transform(X_test)

# **5.  TRAINING THE MODEL**

**During this model training stage, we will select several regression examples:**


---


* LinearRegression()
* RandomForestRegressor()
* KNeighborsRegressor()
* ExtraTreesRegressor()
* xgb.XGBRegressor()
* lgb.LGBMRegressor()
* AdaBoostRegressor()
* GradientBoostingRegressor()
* DecisionTreeRegressor()

**Time Series Regression**
* SARIMAX


---


Additionally, we will explore two neural network models: * MLP and * Sequential from the Keras library, as well as * Sarimax.

Following the predictions and analyzed metrics, we will attempt algorithm tuning and apply L1 L2 regularization to prevent overfitting to the dataset.


---



In [ ]:
# Split data set for SARIMAX Model
train = df_model_ready[:round(len(df_model_ready)*80/100)]
test = df_model_ready[round(len(df_model_ready)*80/100):]
print(test.shape)
print(train.shape)

In [ ]:
# Training model SARIMAX
model = SARIMAX(train['bitcoin_shifted'], order=(2, 1, 2), seasonal_order=(2, 1, 2, 6))
model_fit = model.fit()

# Get the prediction
prediction = model_fit.predict(start=test.index[0], end=test.index[-1])
df_model_ready['predict_sarimax'] = prediction

In [ ]:
# View result metrics
print(f"Mean Squared Error: {np.sqrt(mean_squared_error(test['bitcoin_shifted'], prediction))}")
print(f"Mean Absolute Error: {mean_absolute_error(test['bitcoin_shifted'], prediction)}")
print(f"R 2 Score: {r2_score(test['bitcoin_shifted'], prediction)}")

In [ ]:
# Create copy dataframe and remove  nan values
df_model_ready_remove_nan = df_model_ready.dropna()

# Transform pandas series to DataFrame
df_model_ready_exp = pd.DataFrame({'date': df_model_ready.index, 'bitcoin_shifted': np.exp(df_model_ready['bitcoin_shifted'])})
df_model_ready_remove_nan_exp = pd.DataFrame({'date': df_model_ready_remove_nan.index, 'predict_sarimax': np.exp(df_model_ready_remove_nan['predict_sarimax'])})

# Visualised result prediction
sns.lineplot(data=df_model_ready_exp, x='date', y='bitcoin_shifted', label='bitcoin_shifted')
sns.lineplot(data=df_model_ready_remove_nan_exp, x='date', y='predict_sarimax', label='predict_sarimax')
plt.title('SARIMAX Prediction')



In [ ]:
# We create an object to train a model
linear_regres = LinearRegression()
random_forest = RandomForestRegressor()
knn_regresor = KNeighborsRegressor()
extra_tree = ExtraTreesRegressor()
xg_boost = xgb.XGBRegressor()
lgbm = lgb.LGBMRegressor()
ada_regressor = AdaBoostRegressor()
gradient_regressor = GradientBoostingRegressor()
decision_tree = DecisionTreeRegressor()

# Create list for all object
list_regressor = [linear_regres,
                   random_forest,
                   knn_regresor,
                   extra_tree,
                   xg_boost,
                   lgbm,
                   ada_regressor,
                   gradient_regressor,
                   decision_tree]

In [ ]:
# Create DFrame for append predictions
predictions = pd.DataFrame(y_test).reset_index(drop = True).rename(columns = {0:'btc_shifted'})

In [ ]:
# Create function for apply every model for training
def apply_model(dataframe, to_predict, price, lista_regressor, X_train, y_train):

# We iterate through a for loop for each regression model
  for regressor in tqdm(lista_regressor):
    #We train the model on the training data
    regressor.fit(X_train, y_train)
    # Realised predictions
    y_pred = regressor.predict(to_predict)
# Append the prediction results to the DataFrame under the model name format
    dataframe[f"{str(regressor).split('(')[0]}_{price}"] = y_pred

# Return DFrame
  return dataframe

In [ ]:
# Apply function and create DataFrame with result prediction
df_results_predictions = apply_model(predictions, X_test_scaled, 'btc_shifted', list_regressor, X_train_scaled, y_train)

In [ ]:
# Visualised result prediction , first 5 rows
df_results_predictions.head().T

In [ ]:
# Create a function to inverse exponential values back to their original form for clarity
def series_inverse_exponential(df, column):

    # Create DFrame
    dataframe_predictions_inversed = pd.DataFrame()

# Iterate for each column
    for col in column:
# We return a new NumPy array where each element is the exponential of the corresponding element
        dataframe_predictions_inversed[col] = np.exp(df[col])
# Return DattaFrame
    return dataframe_predictions_inversed

# List with columns
column = ['bitcoin_shifted', 'LinearRegression_btc_shifted',
          'RandomForestRegressor_btc_shifted', 'KNeighborsRegressor_btc_shifted',
          'ExtraTreesRegressor_btc_shifted', 'XGBRegressor_btc_shifted',
          'LGBMRegressor_btc_shifted', 'AdaBoostRegressor_btc_shifted',
          'GradientBoostingRegressor_btc_shifted',
          'DecisionTreeRegressor_btc_shifted']


# Apply function for all data and attribute new DFrame
df_results_predictions_inverse = series_inverse_exponential(df_results_predictions, column)

In [ ]:
# Visualised DataFrame with original values
df_results_predictions_inverse.head().T

In [ ]:
# Evaluate metrics Regressor Model
for column in df_results_predictions.drop(columns = 'bitcoin_shifted').columns:
  print('-' * 70)
  print(column, f"{metrics(df_results_predictions[column], df_results_predictions['bitcoin_shifted'])}")
  print('-' * 70)

In [ ]:
from sklearn.neural_network import MLPRegressor

regressor_model_mlp = MLPRegressor(hidden_layer_sizes=(8, 4, 1),
                                   activation='identity',
                                   solver='adam',
                                   random_state=42,
                                   max_iter=100,
                                   batch_size=16)  # L1 (Lasso) regularization


In [ ]:
# Train model(MLP)
regressor_model_mlp.fit(X_train_scaled, y_train)

In [ ]:

# Calculate prediction
prediction_model_mlp = regressor_model_mlp.predict(X_test_scaled)

# Visualise result metrics
metrics(y_test, prediction_model_mlp)

In [ ]:
def regression_model():

  model = Sequential()
  model.add(Dense(30, activation = 'relu', input_shape = (X_train.shape[1],)))
  model.add(Dense(30, activation = 'relu'))
  model.add(Dense(30, activation = 'relu'))
  model.add(Dense(1))

  model.compile(optimizer = 'adam', loss = 'mean_squared_error')

  return model

In [ ]:
model = regression_model()

In [ ]:
model.fit(X_train_scaled, y_train, epochs = 100)

In [ ]:
predictions_keras = model.predict(X_test_scaled)

In [ ]:
metrics(y_test, predictions_keras)

# 5.1 EVALUATING PERFORMANCE

**Notice**

---



---


Notice that from the neural networks, we obtained one of the best scores:

**MLP:**
- MSE = 0.005025683256722897
- RMSE = 0.0708920535513177
- MAE = 0.05578814706051883
- r2 = 0.8505938429607646

And from the linear regression models, one of the best scores was recorded:

**LinearRegression:**
- MSE = 0.004582210610938365
- RMSE = 0.06769202767636943
- MAE = 0.05127566923314914
- r2 = 0.8627888769554121

**AdaBoostRegressor:**
- MSE = 0.008846475648837839
- RMSE = 0.09405570503078396
- MAE = 0.07268163640788569
- r2 = 0.8054804326559408

The rest of the algorithms recorded a rather weak result... In the next stage, we will apply tuning to them, as things can change.


---



---



   # 5.2  Hyperparameter Tuning

**Notice**



---



---

For each model, we will define a range of parameters to find the best model parameters. Then, these parameters will be applied to the model where we will obtain predictions and evaluate them.


---


The defined parameters are:

-- n_estimators: The number of trees in the Gradient Boosting model.

-- learning_rate: The learning rate, which controls how much each tree contributes to correcting errors from the previous prediction.

-- max_depth: The maximum depth of each tree in the ensemble.

-- subsample: The percentage of random sampling of data in building each tree.

-- colsample_bytree: The percentage of randomly selected features (columns) to build each tree.

-- reg_alpha: The L1 regularization parameter.

-- reg_lambda: The L2 regularization parameter.

-- gamma: A cutting parameter that controls how much the loss should decrease to make a new split on a node.

-- min_samples_split: The minimum number of samples required to split an internal node.

-- min_samples_leaf: The minimum number of samples required to be on a tree leaf.

-- max_features: The maximum number of features to consider when looking for the best split.


---



---







In [ ]:
# We define a parameter grid for each model to determine the optimal combination that maximizes the performance of each model

# Param Tiuning For RandomForest
param_grid_random_forest = {
                            'n_estimators': [100, 200, 300],
                            'max_depth': [None, 10, 20],
                            'min_samples_split': [2, 5, 10],
                            'min_samples_leaf': [1, 2, 4],
                            'max_features': ['auto', 'sqrt']
                            }

# Param Tiuning For KNN
param_grid_knn = {'n_neighbors': [3, 5, 7, 10]}

# Param Tiuning For ExtraTree
param_grid_extra_tree = {
                        'n_estimators': [100, 200, 300],
                        'max_depth': [None, 10, 20],
                        'min_samples_split': [2, 5, 10],
                        'min_samples_leaf': [1, 2, 4],
                        'max_features': ['auto', 'sqrt']
                        }

# Param Tiuning For XGBoost
param_grid_xg_boost = {
                      'n_estimators': [100, 200, 300],
                      'learning_rate': [0.05, 0.1, 0.2],
                      'max_depth': [3, 4, 5],
                      'subsample': [0.8, 0.9, 1.0],
                      'colsample_bytree': [0.8, 0.9, 1.0],
                      'gamma': [0, 0.1, 0.2]
                      }

# Param Tiuning For LGBM
param_grid_lgbm = {
                  'n_estimators': [100, 200, 300],
                  'learning_rate': [0.05, 0.1, 0.2],
                  'max_depth': [3, 4, 5],
                  'subsample': [0.8, 0.9, 1.0],
                  'colsample_bytree': [0.8, 0.9, 1.0],
                  'reg_alpha': [0, 0.1, 0.2],
                  'reg_lambda': [0, 0.1, 0.2]
                  }

# Param Tiuning For AdaRegressor
param_grid_ada_regressor = {
                            'n_estimators': [50, 100, 200],
                            'learning_rate': [0.05, 0.1, 0.2]
                            }

# Param Tiuning For GradientRegressor
param_grid_gradient_regressor = {
                                'n_estimators': [50, 100, 200],
                                'learning_rate': [0.05, 0.1, 0.2],
                                'max_depth': [3, 4, 5],
                                'subsample': [0.8, 0.9, 1.0]
                                }

# Param Tiuning For DecisionTree
param_grid_decision_tree = {
                            'max_depth': [None, 10, 20],
                            'min_samples_split': [2, 5, 10],
                            'min_samples_leaf': [1, 2, 4],
                            'max_features': ['auto', 'sqrt']
                            }

In [ ]:
# Performs grid search with cross-validation for every model

# Random Forest
grid_search_random_forest = GridSearchCV(estimator = random_forest,
                                         param_grid=param_grid_random_forest,
                                         cv=5,
                                         n_jobs=-1,
                                         verbose=2)

# KNN
grid_search_knn = GridSearchCV(estimator = knn_regresor,
                               param_grid=param_grid_knn,
                               cv=5,
                               n_jobs=-1,
                               verbose=2)

# ExtraTree
grid_search_extra_tree = GridSearchCV(estimator = extra_tree,
                                      param_grid=param_grid_extra_tree,
                                      cv=5,
                                      n_jobs=-1,
                                      verbose=2)

# XGBoost
grid_search_xgboost = GridSearchCV(estimator = xg_boost,
                                   param_grid=param_grid_xg_boost,
                                   cv=5,
                                   n_jobs=-1,
                                   verbose=2)

# LGBM
grid_search_lgbm = GridSearchCV(estimator = lgbm,
                                param_grid=param_grid_lgbm,
                                cv=5,
                                n_jobs=-1,
                                verbose=2)

# AdaRegressor
grid_search_ada_regressor = GridSearchCV(estimator = ada_regressor,
                                         param_grid=param_grid_ada_regressor,
                                         cv=5,
                                         n_jobs=-1,
                                         verbose=2)

# Gradient Regressor
grid_search_gradient_regressor = GridSearchCV(estimator = gradient_regressor,
                                              param_grid=param_grid_gradient_regressor,
                                              cv=5,
                                              n_jobs=-1,
                                              verbose=2)

# DecisonTree
grid_search_decision_tree = GridSearchCV(estimator = decision_tree,
                                         param_grid=param_grid_decision_tree,
                                         cv=5,
                                         n_jobs=-1,
                                         verbose=2)

In [ ]:
# Dentifies the best set of hyperparameters that optimize the models performance based on the specified scoring metric
grid_search_random_forest.fit(X_train_scaled, y_train)
grid_search_knn.fit(X_train_scaled, y_train)
grid_search_extra_tree.fit(X_train_scaled, y_train)
grid_search_xgboost.fit(X_train_scaled, y_train)
grid_search_lgbm.fit(X_train_scaled, y_train)
grid_search_ada_regressor.fit(X_train_scaled, y_train)
grid_search_gradient_regressor.fit(X_train_scaled, y_train)
grid_search_decision_tree.fit(X_train_scaled, y_train)

In [ ]:
# Print best parametres for every model

print(f"Best Parametrs for  Randomn Forest: {grid_search_random_forest.best_params_}")
print(f"Best Parametrs for  KNN: {grid_search_knn.best_params_}")
print(f"Best Parametrs for  Extra Tree: {grid_search_extra_tree.best_params_}")
print(f"Best Parametrs for  XGBoost: {grid_search_xgboost.best_params_}")
print(f"Best Parametrs for  LGBM: {grid_search_lgbm.best_params_}")
print(f"Best Parametrs for  ADA: {grid_search_ada_regressor.best_params_}")
print(f"Best Parametrs for  Gradient: {grid_search_gradient_regressor.best_params_}")
print(f"Best Parametrs for  Decision Tree: {grid_search_decision_tree.best_params_}")

In [ ]:
# Initialization with the best parameters identified in the grid search process

# Stocking every best parameters
best_params_1 = grid_search_random_forest.best_params_
# Apply parametres for RandomForestRegressor
regresor_forest = RandomForestRegressor(n_estimators=best_params_1['n_estimators'],
                                       max_depth=best_params_1['max_depth'],
                                       min_samples_split=best_params_1['min_samples_split'],
                                       min_samples_leaf=best_params_1['min_samples_leaf'],
                                       max_features=best_params_1['max_features'])

# Stocking every best parameters
best_params_2 = grid_search_knn.best_params_
# Apply parametres for KNeighborsRegressor
regresor_knn = KNeighborsRegressor(n_neighbors=best_params_2['n_neighbors'])


# Stocking every best parameters
best_params_3 = grid_search_extra_tree.best_params_
# Apply parametres for ExtraTreesRegressor
regresor_extra_tree = ExtraTreesRegressor(n_estimators=best_params_3['n_estimators'],
                                        max_depth=best_params_3['max_depth'],
                                        min_samples_split=best_params_3['min_samples_split'],
                                        min_samples_leaf=best_params_3['min_samples_leaf'],
                                        max_features=best_params_3['max_features'])

# Stocking every best parameters
best_params_4 = grid_search_xgboost.best_params_
# Apply parametres for XGBRegressor
regresor_xgboost = xgb.XGBRegressor(n_estimators=best_params_4['n_estimators'],
                                    max_depth=best_params_4['max_depth'],
                                    learning_rate=best_params_4['learning_rate'],
                                    subsample=best_params_4['subsample'],
                                    colsample_bytree=best_params_4['colsample_bytree'],
                                    gamma=best_params_4['gamma'])

# Stocking every best parameters
best_params_5 = grid_search_lgbm.best_params_
# Apply parametres for LGBMRegressor
regresor_lgbm = lgb.LGBMRegressor(n_estimators=best_params_5['n_estimators'],
                                  max_depth=best_params_5['max_depth'],
                                  learning_rate=best_params_5['learning_rate'],
                                  subsample=best_params_5['subsample'],
                                  colsample_bytree=best_params_5['colsample_bytree'],
                                  reg_alpha=best_params_5['reg_alpha'],
                                  reg_lambda=best_params_5['reg_lambda'])

# Stocking every best parameters
best_params_6 = grid_search_ada_regressor.best_params_
# Apply parametres for AdaBoostRegressor
regresor_ada = AdaBoostRegressor(n_estimators=best_params_6['n_estimators'],
                                    learning_rate=best_params_6['learning_rate'])

# Stocking every best parameters
best_params_7 = grid_search_gradient_regressor.best_params_
# Apply parametres for GradientBoostingRegressor
regresor_gradient = GradientBoostingRegressor(n_estimators=best_params_7['n_estimators'],
                                              learning_rate=best_params_7['learning_rate'],
                                              max_depth=best_params_7['max_depth'])

# Stocking every best parameters
best_params_8 = grid_search_decision_tree.best_params_
# Apply parametres for DecisionTreeRegressor
regresor_decision_tree = DecisionTreeRegressor(max_depth=best_params_8['max_depth'],
                                              min_samples_split=best_params_8['min_samples_split'],
                                              min_samples_leaf=best_params_8['min_samples_leaf'],
                                              max_features=best_params_8['max_features'])

In [ ]:
# Train the models on the data

regresor_forest.fit(X_train_scaled, y_train)
regresor_knn.fit(X_train_scaled, y_train)
regresor_extra_tree.fit(X_train_scaled, y_train)
regresor_xgboost.fit(X_train_scaled, y_train)
regresor_lgbm.fit(X_train_scaled, y_train)
regresor_ada.fit(X_train_scaled, y_train)
regresor_gradient.fit(X_train_scaled, y_train)
regresor_decision_tree.fit(X_train_scaled, y_train)

In [ ]:
#  Apply and attribute predictions for the test column

prediction_regresor_forest = regresor_forest.predict(X_test_scaled)
prediction_regresor_knn = regresor_knn.predict(X_test_scaled)
prediction_regresor_extra_tree = regresor_extra_tree.predict(X_test_scaled)
prediction_regresor_xgboost = regresor_xgboost.predict(X_test_scaled)
prediction_regresor_lgbm = regresor_lgbm.predict(X_test_scaled)
prediction_regresor_ada = regresor_ada.predict(X_test_scaled)
prediction_regresor_gradient = regresor_gradient.predict(X_test_scaled)
prediction_regresor_decision_tree = decision_tree.predict(X_test_scaled)

In [ ]:
# Print result metrics for every tuned models

print('-' * 70)
print(f'Forest Tiuning: {metrics(y_test, prediction_regresor_forest)}')
print('-' * 70)
print(f'KNN Tiuning: {metrics(y_test, prediction_regresor_knn)}')
print('-' * 70)
print(f'Extra Tree Tiunig: {metrics(y_test, prediction_regresor_extra_tree)}')
print('-' * 70)
print(f'Gradient Boosting: {metrics(y_test, prediction_regresor_xgboost)}')
print('-' * 70)
print(f'LGBM Tiuning: {metrics(y_test, prediction_regresor_lgbm)}')
print('-' * 70)
print(f'ADA Tiuning: {metrics(y_test, prediction_regresor_ada)}')
print('-' * 70)
print(f'Gradient Tiuning: {metrics(y_test, prediction_regresor_gradient)}')
print('-' * 70)
print(f'Decision Tree Tiuning: {metrics(y_test, prediction_regresor_decision_tree)}')
print('-' * 70)

# 5.3 Apply penalty L1 L2 Neural Model

In [ ]:
# Set penalty parametres for MLP model
model_mlp_regularized = MLPRegressor(hidden_layer_sizes=(8, 4, 2),
                                   activation='identity',
                                   solver='adam',
                                   random_state=42,
                                   max_iter=500,
                                   batch_size=16,
                                   alpha=0.001,  # for L2 (Ridge)
                                   beta_1=0.9,   # forr L1 (Lasso)
                                   beta_2=0.999)  # for L1 (Lasso)

In [ ]:
# Train model
model_mlp_regularized.fit(X_train_scaled, y_train)

In [ ]:
# Get prediction
prediction_model_mlp_regularized = model_mlp_regularized.predict(X_test_scaled)

In [ ]:
# View metrics
metrics(y_test, prediction_model_mlp_regularized)

In [ ]:
# Set penalty parametres for Sequential model

def regression_model_regularized():

    model = Sequential()

    model.add(Dense(25, activation='relu', input_shape=(X_train.shape[1],),
                    kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01),
                    bias_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01)))

    model.add(Dense(25, activation='relu',
                    kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01),
                    bias_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01)))

    model.add(Dense(25, activation='relu',
                    kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01),
                    bias_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01)))

    model.add(Dense(1))

    model.compile(optimizer='adam', loss='mean_squared_error')

    return model


In [ ]:
# Call function
model_regularized_keras = regression_model_regularized()

In [ ]:
# trainning model
model_regularized_keras.fit(X_train_scaled, y_train, epochs=100)

In [ ]:
# Get prediction
prediction_model_regularized_keras = model_regularized_keras.predict(X_test_scaled)

In [ ]:
# View metrics
metrics(y_test, prediction_model_regularized_keras)

**Notice**



---
Observing that tuning the regression models did not improve things, following this manipulation, we obtained slightly better results, but one can still consider the outcomes to be weak. However, penalizing the neural networks helped us improve the metrics.

For the MLP:

* MSE = 0.0038460050884882822
* RMSE = 0.062016167960365645
* MAE = 0.045336229539834315
* r2 = 0.8856639364496938

And for Sequential, the best score:

* MSE = 0.0025708550747888825
* RMSE = 0.050703600215259687
* MAE = 0.0373461048751258
* r2 = 0.9235722672105913


# **6.  FINAL EVALUATION AND INTERPRETATION OF RESULT**

In [ ]:
visualised_best_model = np.exp(prediction_model_regularized_keras)

In [ ]:
# Create function for visualize the prediction on a line plot

def visualised_prediction(df, prediction):

# Defined size image
    plt.figure(figsize=(16, 6))
    plt.style.use('seaborn-darkgrid')
    # Filtred time period of  interest
    ds_filtered = df[df.index >= pd.Timestamp('2020-01-01')]
# We plot the filtered data with the actual past values
    plt.plot(ds_filtered.index[:-len(prediction)],
            ds_filtered['btc_shifted'][:-len(prediction)],
            label='Real Bitcoin Price', color='green')
# We plot the  data with the prediction values
    plt.plot(df.index[-len(prediction):], prediction,
            label='Predicted Bitcoin Price', color='red', linestyle='-', linewidth=2)
    bub_size = 100
    sns.scatterplot(data=data, x='data', y='btc_price', hue='psihologi_cicle', palette=pal, s=bub_size)

# Set title and name for axes
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price')
    plt.title('Layering Charts: Real vs. Predicted Bitcoin Prices Compared to Human Psychology', fontsize=16)
    plt.legend()
    plt.grid(axis='x')
# Show image
    plt.show()


# Call function
visualised_prediction(data_preprocessing, visualised_best_model)


**Notice**

---

As a result, we can observe that investors are in a psychological state of hope. Additionally, we can see that the BTC price represents an upward trend, which could encourage investors to accelerate the price growth. Furthermore, we can observe that BTC has an accumulation range between the prices of 41,000 and 47,000, indicating that we have chances to move upwards


---




In [ ]:
# Create DataFrame with column Bitcoin Price
df_temporal = data[['btc_price', 'data']]


In [ ]:
# Create New DataFrame with prediction
df_predictions = pd.DataFrame({'data': data_preprocessing.index[-len(visualised_best_model):]}).reset_index(drop=True)
df_predictions['Prediction'] = visualised_best_model



In [ ]:
# We join the dataframe with the target column and the dataframe with the prediction column
df_result_final = pd.merge(df_predictions, df_temporal, on='data', how='left')

In [ ]:
# We set column 'data' as index
df_result_final = df_result_final.set_index('data')

# View the last 20 rows
df_result_final.tail(20)

In [ ]:
# View real price BTC for the last 30 days
df_result_final['btc_price'].plot(kind='line', figsize=(10, 4), title='btc_price')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# View predicted price (Sequential neuronal model) BTC for the last 30 days
df_result_final['Prediction'].plot(kind='line', figsize=(10, 4), title='Prediction')
plt.gca().spines[['top', 'right']].set_visible(False)


6. **FINAL EVALUATION AND INTERPRETATION OF RESULT**



---



• Through these two line charts, we can observe that the prediction made by the neural network shows a slight deviation from the actual price of Bitcoin. This is also confirmed by the table that presents two columns: one with the real price and another with the predicted price. Additionally, we can see that the model made fairly accurate predictions on some days, with a deviation of 1.08% from the real price, while on other days, we observe a deviation of up to 6.19%, which represents a quite significant error.
• To improve the model's predictions, it is necessary to use newer and more accurate data, as well as a more detailed examination and meticulous adjustment of the algorithms to improve the precision of price analysis.

For investors, predictions can provide valuable guidance in making investment decisions, helping them identify trading opportunities and manage the risks associated with Bitcoin price volatility.

For traders, models can serve as essential tools in optimizing their trading strategies, allowing them to conduct more efficient and profitable trades.

Other entities involved in the Bitcoin market, such as trading platforms or investment funds, can benefit from predictions to plan and manage the risks associated with their exposure to this volatile asset. In conclusion, prediction models are essential tools for portfolio management and for optimizing investment performance in the complex landscape of cryptocurrencies.


In addition, it is important to emphasize that, besides the rigorous analysis of algorithms and prediction models, it is essential to consider the context of the crypto market. Volatility and abrupt changes are common aspects of the Bitcoin market, and these can influence the outcomes of our models. Therefore, it is advisable to be attentive to developments and events in the industry and to adapt our models and strategies accordingly.

Furthermore, collaboration and information sharing among various entities involved in the Bitcoin market can contribute to a more comprehensive understanding of market dynamics and the development of more robust portfolio management and risk mitigation strategies.

In a constantly evolving world, prediction models are an essential tool, but flexibility and adaptability are equally important for success in such a dynamic landscape as cryptocurrencies.


---


